# Data Science Bootcamp 2026_1 / Тестовое задание / Classic ML

Выполнил Махлин Мирон

## Общее описание подхода

Решение использует CatBoost-модели для итогового предсказания скора и специализированый подход для валидации и оценки качества модели.

Валидация:
- Используется временная кросс-валидация с 5 фолдами: модель обучается на префиксе данных и предсказывает для следующего небольшого куска для пяти увеличивающихся префиксов. Это соответствует тому, как устроено обучение и инференс модели в проде.
- В качестве метрики используется Daily Average Precision, т.к. именно она является целевой метрикой соревнования.
- При этом для генерации признаков используются лишь события, произошедшие до даты назначения, чтобы избежать утечек.

Ключевые используемые признаки:
- Счётчки событий и периоды (уникальные типы, активные дни и тп)
- Ценовые статистики
- Контекстные признаки
- Признаки последнего события (тип, контекст)
- Скользящие признаки за последние 24 часа
- Отношения (избранное к просмотрам, переписки к звонкам, поиск к просмотрам)
- Анализ False Positives показал, что модель иногда переобучается на "сырую активность": молодые пользователи с огромным числом просмотров получают высокий скор, хотя не конвертируются. Анализ False Negatives выявил, что взрослые аккаунты с низкой недавней активностью иногда конвертируются, но получают низкий скор. Для компенсации этого был добавлен признак отношения недавней активности к исторической (recent_activity_ratio).
- _Что пробовал, но не помогло_: кросс-фичи цен (assign price vs user avg view price) привели к падению метрики; удаление слабозначимых фичей также слегка ухудшило генерализацию ансамбля

Модель:
- Используется CatBoostClassifier
- Сравнивал CatBoost, XGBoost и LightGBM. CatBoost показал метрику DAP на 50-60% выше (0.63 против 0.42 у XGBoost). Вероятно, это связано с нативной обработкой множества категориальных признаков
- Отключил веса классов (auto_class_weights). Поскольку целевой метрикой является Average Precision (ранжирование), сырые вероятности работают лучше балансировки. Это дало прирост около 3% к метрике.
- Оптимизировал параметры: лучшим сочетанием оказались depth=7 и learning_rate=0.03
- Используются эмперически подобранные гиперпараметры, которые опционально могут быть оптимизированы с помощью Optuna
- Для итогового предсказания используется ансамбль из пяти моделей с разными сидами, что позволяеть уменьшить дисперсию и увеличить стабильность решения

Воспроизводимость:
- 42 в качестве RANDOM SEED
- Фиксация RANDOM STATE на протяжении всего пайплайна

## Конфигурация

Делаем все импорты, задаём параметры модели, имена колонок и другие константы, которые будем использовать в дальнейшем.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier, Pool
from tqdm.auto import tqdm
import optuna
import logging
import random
import os
from config import (
    RANDOM_STATE, ROOT, DATA_DIR, TARGET, NON_FEATURE_COLUMNS,
    ENSEMBLE_SEEDS, MODEL_CONFIG, EVENT_FEATURE_COLS, EVENT_COUNT_COLS, seed_everything
)

logging.getLogger("optuna").setLevel(logging.INFO)
seed_everything(RANDOM_STATE)
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)

## Утилиты для событийных фичей

Задаём различные хелперы, которые будем использовать в дальнейшем для извлечения полезных признаков из events.csv (счётчики, временные паттерны, статистики цен, последовательности контекста, признаки давности событий и 24-часовые окна). _Примечание: исходный код см. в features.py_

In [2]:
from features import extract_event_features

## Загрузка данных

In [3]:
print("Загрузка данных...")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
events = pd.read_csv(DATA_DIR / "events.csv")

print(f"train: {train.shape}")
print(f"test: {test.shape}")
print(f"events: {events.shape}")

Загрузка данных...
train: (13694, 119)
test: (4306, 118)
events: (254705, 7)


## Подсчёт событийных признаков

Считаем событийные признаки для трейна и теста, пользуясь функциями, заданными выше.

In [4]:
print("Извлечение событийных признаков...")
event_features_train = extract_event_features(events, train)
event_features_test = extract_event_features(events, test)

event_feature_cols = [col for col in event_features_train.columns if col != 'lead_id']

train = train.merge(event_features_train, on='lead_id', how='left')
test = test.merge(event_features_test, on='lead_id', how='left')

for col in event_feature_cols:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

print(f"Добавлены событийные признаки: {len(event_feature_cols)} штук")

Извлечение событийных признаков...
Добавлены событийные признаки: 39 штук


## Разделение признаков

In [5]:
feature_columns = [
    column for column in train.columns
    if column not in NON_FEATURE_COLUMNS and column in test.columns
]

numeric_columns = [
    column for column in feature_columns
    if pd.api.types.is_numeric_dtype(train[column])
]

categorical_columns = [
    column for column in feature_columns
    if column not in numeric_columns
]

print(f"Всего признаков: {len(feature_columns)}")
print(f"Числовых: {len(numeric_columns)}")
print(f"Категориальных: {len(categorical_columns)}")

Всего признаков: 153
Числовых: 144
Категориальных: 9


## Подготавливаем категориальные признаки

Для совместимости с CatBoost заменяем пропуски на строку "\_\_nan_\_" (они будут трактоваться как отдельная категория).

In [6]:
def prepare_for_catboost(df, categorical_columns):
    df = df.copy()
    for col in categorical_columns:
        df[col] = df[col].astype("string").fillna("__nan__").astype(str)
    return df

train = prepare_for_catboost(train, categorical_columns)
test = prepare_for_catboost(test, categorical_columns)

Имплементируем целевую метрику DAP. _Примечание: исходный код см. в файле metrics.py_

In [7]:
from metrics import daily_average_precision

Подготавливаем функцию для оптимизации гиперпараметров при помощи оптуны.

In [8]:
from metrics import run_cv_fold

def optimize_hyperparameters(
    train_df,
    feature_columns,
    categorical_columns,
    target_column,
    base_config,
    n_trials=25,
    n_splits=3,
):
    dates = pd.to_datetime(train_df["assignment_date"]).dt.date
    ordered_dates = np.array(sorted(dates.unique()))
    n_dates = len(ordered_dates)
    start_idx = int(n_dates * 0.5)
    fold_edges = np.linspace(start_idx, n_dates, n_splits + 1).astype(int)
    fold_edges = np.unique(fold_edges)
    
    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 6, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        }
        
        model_config = base_config.copy()
        model_config.update(params)
        
        scores = []
        for fold in range(len(fold_edges) - 1):
            lo, hi = fold_edges[fold], fold_edges[fold + 1]
            if lo >= hi:
                continue
            
            cutoff_train = ordered_dates[lo]
            cutoff_valid_end = ordered_dates[hi - 1]
            
            train_mask = dates < cutoff_train
            valid_mask = (dates >= cutoff_train) & (dates <= cutoff_valid_end)
            
            train_fold = train_df[train_mask].copy()
            valid_fold = train_df[valid_mask].copy()
            
            if len(train_fold) == 0 or len(valid_fold) == 0:
                continue
            if valid_fold[target_column].sum() == 0:
                continue
            
            fold_daily_ap, _ = run_cv_fold(
                train_fold, valid_fold, feature_columns, categorical_columns, target_column, model_config
            )
            scores.append(fold_daily_ap)
        
        return np.mean(scores) if scores else 0.0
    
    study = optuna.create_study(direction="maximize", study_name="catboost_lead_scoring")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    print(f"Лучший запуск: {study.best_trial.number}, скор: {study.best_value:.5f}")
    print(f"Лучшие параметры: {study.best_params}")
    
    optimized_config = base_config.copy()
    optimized_config.update(study.best_params)
    
    return optimized_config, study

## Временная кросс-валидация

Для более точного оценки модели на исторических данных используем временную кросс-валидацию с корректными временными разбиениями.

In [9]:
def time_based_cv_catboost(
    train_df,
    feature_columns,
    categorical_columns,
    target_column,
    model_config,
    date_column="assignment_date",
    n_splits=5,
    min_train_frac=0.5,
    skip_days_without_positives: bool = True,
    verbose: bool = True,
):
    dates = pd.to_datetime(train_df[date_column]).dt.date
    ordered_dates = np.array(sorted(dates.unique()))
    n_dates = len(ordered_dates)

    start_idx = int(n_dates * min_train_frac)
    fold_edges = np.linspace(start_idx, n_dates, n_splits + 1).astype(int)
    fold_edges = np.unique(fold_edges)

    results = []
    all_per_day = []

    fold_range = range(len(fold_edges) - 1)
    for fold in tqdm(fold_range, desc="CV folds", disable=not verbose):
        lo, hi = fold_edges[fold], fold_edges[fold + 1]
        if lo >= hi:
            continue

        cutoff_train = ordered_dates[lo]
        cutoff_valid_end = ordered_dates[hi - 1]

        train_mask = dates < cutoff_train
        valid_mask = (dates >= cutoff_train) & (dates <= cutoff_valid_end)

        train_fold = train_df[train_mask].copy()
        valid_fold = train_df[valid_mask].copy()

        if len(train_fold) == 0 or len(valid_fold) == 0:
            continue
        if valid_fold[target_column].sum() == 0:
            continue

        train_pool = Pool(
            train_fold[feature_columns],
            train_fold[target_column],
            cat_features=categorical_columns,
        )
        valid_pool = Pool(
            valid_fold[feature_columns],
            valid_fold[target_column],
            cat_features=categorical_columns,
        )

        fold_model = CatBoostClassifier(**model_config)
        fold_model.fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=False)

        valid_scores = fold_model.predict_proba(valid_fold[feature_columns])[:, 1]

        fold_daily_ap, per_day = daily_average_precision(
            y_true=valid_fold[target_column],
            y_score=valid_scores,
            assignment_date=valid_fold[date_column],
            skip_days_without_positives=skip_days_without_positives,
            return_per_day=True,
        )
        fold_overall_ap = average_precision_score(valid_fold[target_column], valid_scores)

        results.append({
            "fold": fold,
            "train_size": len(train_fold),
            "valid_size": len(valid_fold),
            "valid_start": cutoff_train,
            "valid_end": cutoff_valid_end,
            "best_iteration": fold_model.get_best_iteration(),
            "daily_ap": fold_daily_ap,
            "overall_ap": fold_overall_ap,
            "n_valid_days": len(per_day),
        })
        all_per_day.append(per_day)

    results_df = pd.DataFrame(results)

    summary = {
        "mean_daily_ap": float(results_df["daily_ap"].mean()) if len(results_df) else float("nan"),
        "std_daily_ap": float(results_df["daily_ap"].std()) if len(results_df) else float("nan"),
        "mean_overall_ap": float(results_df["overall_ap"].mean()) if len(results_df) else float("nan"),
        "mean_best_iteration": float(results_df["best_iteration"].mean()) if len(results_df) else float("nan"),
        "n_folds": len(results_df),
    }

    return summary, results_df, all_per_day

Опционально запускаем оптуну для подбора гиперпараметров (может занять несколько часов).

In [10]:
# MODEL_CONFIG, study = optimize_hyperparameters(
#     train_df=train,
#     feature_columns=feature_columns,
#     categorical_columns=categorical_columns,
#     target_column=TARGET,
#     base_config=MODEL_CONFIG,
#     n_trials=25,
#     sampler=sampler,
#     n_splits=3,
# )

Запускаем кросс-валидацию, а также обычную валидацию на 80% выборки.

In [11]:
print("\nКросс-валидация")

summary, results_df, per_day_list = time_based_cv_catboost(
    train_df=train,
    feature_columns=feature_columns,
    categorical_columns=categorical_columns,
    target_column=TARGET,
    model_config=MODEL_CONFIG,
    n_splits=5,
    min_train_frac=0.5,
)

print(results_df.to_string(index=False))
print(summary)

tail_daily_ap = results_df.sort_values("valid_start").tail(3)["daily_ap"].mean()
print(f"\nСредняя DAP на трёх последних фолдах: {tail_daily_ap:.5f}")

print("\nФинальная кроссвалидация")

summary, results_df, per_day_list = time_based_cv_catboost(
    train_df=train,
    feature_columns=feature_columns,
    categorical_columns=categorical_columns,
    target_column=TARGET,
    model_config=MODEL_CONFIG,
    n_splits=1,
    min_train_frac=0.8,
)

print(results_df.to_string(index=False))
print(summary)


Кросс-валидация


CV folds:   0%|          | 0/5 [00:00<?, ?it/s]

Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


 fold  train_size  valid_size valid_start  valid_end  best_iteration  daily_ap  overall_ap  n_valid_days
    0        6743         899  2026-04-15 2026-04-15             602  0.628202    0.628202             1
    1        7642        1755  2026-04-16 2026-04-17             924  0.683795    0.686336             2
    2        9397         875  2026-04-18 2026-04-18             538  0.646661    0.646661             1
    3       10272        1724  2026-04-19 2026-04-20            1141  0.707137    0.707027             2
    4       11996        1698  2026-04-21 2026-04-22            1925  0.684079    0.681741             2
{'mean_daily_ap': 0.6699746426743496, 'std_daily_ap': 0.03185696827516211, 'mean_overall_ap': 0.6699934183408051, 'mean_best_iteration': 1026.0, 'n_folds': 5}

Средняя DAP на трёх последних фолдах: 0.67929

Финальная кроссвалидация


CV folds:   0%|          | 0/1 [00:00<?, ?it/s]

Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


 fold  train_size  valid_size valid_start  valid_end  best_iteration  daily_ap  overall_ap  n_valid_days
    0       10272        3422  2026-04-19 2026-04-22            1412  0.693346      0.6906             4
{'mean_daily_ap': 0.6933464737214786, 'std_daily_ap': nan, 'mean_overall_ap': 0.690600121164128, 'mean_best_iteration': 1412.0, 'n_folds': 1}


## Финальное обучение и предсказание

Обучаем 5 моделей CatBoost с разными сидами и усредняем предсказания для снижения дисперсии при сохранении смещения.

In [12]:
final_iterations = int(round(summary["mean_best_iteration"]))
final_iterations = max(final_iterations, 100)

print(f"\nАнсамблируем {len(ENSEMBLE_SEEDS)} моделей, используя {final_iterations} итераций...")
print(f"Сиды: {ENSEMBLE_SEEDS}")

final_pool = Pool(
    train[feature_columns],
    train[TARGET],
    cat_features=categorical_columns,
)

test_scores_list = []
for i, seed in enumerate(tqdm(ENSEMBLE_SEEDS, desc="Ансамбль")):
    ensemble_config = MODEL_CONFIG.copy()
    ensemble_config["iterations"] = final_iterations
    ensemble_config["random_seed"] = seed
    ensemble_config.pop("early_stopping_rounds", None)
    
    ensemble_model = CatBoostClassifier(**ensemble_config)
    ensemble_model.fit(final_pool, verbose=False)
    
    test_scores_list.append(ensemble_model.predict_proba(test[feature_columns])[:, 1])

test_scores = np.mean(test_scores_list, axis=0)
print(f"\nАнсамблирование завершено")

feature_importance = ensemble_model.get_feature_importance()
importance_df = pd.DataFrame({
    'feature': feature_columns,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("\nТоп-20 самых полезных признаков")
print(importance_df.head(20).to_string(index=False))

print("\nТоп-20 самых бесполезных признаков")
print(importance_df.tail(20).to_string(index=False))

zero_importance = importance_df[importance_df['importance'] == 0]
if len(zero_importance) > 0:
    print(f"\nПризнаки с нулевой важностью:")
    print(zero_importance['feature'].tolist())


Ансамблируем 5 моделей, используя 1412 итераций...
Сиды: [42, 179, 57, 91, 1543]


Ансамбль:   0%|          | 0/5 [00:00<?, ?it/s]

Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time



Ансамблирование завершено

Топ-20 самых полезных признаков
                feature  importance
         events_ctx_c03    8.559659
     events_price_range    7.807192
  seller_page_views_14d    5.205153
  seller_page_views_30d    4.597266
   seller_page_views_7d    3.742298
        last_event_type    3.085904
            lead_source    2.827850
       search_views_90d    2.490983
         events_ctx_c07    1.959184
           last_ctx_seq    1.833672
       photo_swipes_90d    1.816442
   hours_since_last_c03    1.783598
      user_contacts_30d    1.518361
         item_views_14d    1.482348
         events_ctx_c05    1.440530
events_hours_since_last    1.411354
        events_min_slot    1.346394
           events_total    1.284178
     detail_expands_90d    1.218368
similar_item_clicks_90d    1.137129

Топ-20 самых бесполезных признаков
                  feature  importance
          photo_swipes_1d    0.073021
        detail_expands_1d    0.072236
           call_clicks_7d    0.071

Наибольший вклад в модель вносят тип и контекст последнего действия пользователя перед назначением (c03, last_event_type), его ценовое поведение и среднесрочная история активности (7-30 дней). При этом готовые табличные агрегаты за короткие окна (1-3 дня) и метрики прошлых назначений оказались фактически шумовыми, поскольку сиюминутные намерения пользователя гораздо точнее перехватывается нашими сгенерированными событийными признаками. 

## Создаём и валидируем посылку

In [13]:
submission = pd.DataFrame({
    "lead_id": test["lead_id"].astype(str),
    "score": test_scores,
})


assert list(submission.columns) == ["lead_id", "score"], "Invalid columns"
assert len(submission) == len(test), "Invalid row count"
assert submission["lead_id"].is_unique, "Duplicate lead_ids"
assert submission["score"].between(0, 1).all(), "Scores out of range [0, 1]"

print("Все проверки пройдены")

submission.to_csv(ROOT / "submission.csv", index=False)
print(f"\nСоздана submission.csv с {len(submission)} предсказаниями")
print(f"Диапазон: [{submission['score'].min():.4f}, {submission['score'].max():.4f}]")
print(f"Среднее: {submission['score'].mean():.4f}")

Все проверки пройдены

Создана submission.csv с 4306 предсказаниями
Диапазон: [0.0015, 0.9995]
Среднее: 0.2084
